# Notebook Acadêmico: Predição de Churn em Telecomunicações
## Fase 03: Pré-processamento, SMOTE, Modelagem com XGBoost e Avaliação Crítica

---

### 1. Contexto do Experimento
Este experimento aborda o desafio de redução de evasão de clientes (*churn*) em uma companhia telefônica.
O objetivo prático é construir, validar e interpretar um modelo preditivo capaz de identificar clientes com maior probabilidade de cancelamento, fornecendo subsídios para ações preventivas de retenção.

### 2. Objetivos Técnicos e Acadêmicos
#### 2.1 Objetivo Geral
Produzir um notebook acadêmico reprodutível que documente todo o pipeline de Machine Learning para previsão de churn, desde o carregamento dos dados até a avaliação crítica dos resultados obtidos.

#### 2.2 Objetivos Específicos
- Reproduzir o experimento em ambiente local de desenvolvimento.
- Carregar e validar o conjunto de dados utilizado no estudo.
- Aplicar limpeza de dados e tratamento de valores ausentes.
- Separar corretamente variáveis explicativas e variável-alvo.
- Identificar variáveis numéricas e categóricas.
- Construir um pipeline de pré-processamento com imputação, padronização e One-Hot Encoding.
- Realizar a divisão estratificada manual (70% treino / 30% teste).
- Tratar o desbalanceamento da classe churn com SMOTE.
- Treinar um modelo XGBoost para classificação.
- Avaliar o modelo com métricas adequadas a dados desbalanceados (ROC-AUC, PR-AUC, Recall, Precision, F1-score).
- Formular insights e estratégias práticas de negócio para mitigação de churn.
- Documentar limitações, conclusões e próximos passos.

### 3. Descrição da Base de Dados
O conjunto de dados (`telecom_churn_synthetic.csv`) contém 6.000 registros e 27 colunas com as seguintes características:
- **Total de registros:** 6.000
- **Total de colunas:** 27
- **Variável-alvo:** `churn` (0: cliente mantido [5.157 registros], 1: cliente cancelado [843 registros], taxa ~14,05%)
- **Identificador a ser removido:** `customer_id`

**Dicionário de Variáveis:**
- **Identificação:** `customer_id`
- **Perfil e Contrato:** `gender`, `senior_citizen`, `partner`, `dependents`, `tenure_months`, `contract`
- **Serviços Contratados:** `phone_service`, `multiple_lines`, `internet_service`, `online_security`, `online_backup`, `device_protection`, `tech_support`, `streaming_tv`, `streaming_movies`
- **Cobrança e Pagamento:** `paperless_billing`, `payment_method`, `monthly_charges`, `total_charges`, `late_payments_last_6m`, `promo_active`, `discount_pct`
- **Qualidade e Relacionamento:** `complaints_last_3m`, `outages_last_3m`, `avg_download_mbps`
- **Alvo:** `churn`

### 4. Importação de Bibliotecas e Configuração do Ambiente
Importação dos módulos necessários para manipulação de dados, visualização, pré-processamento, balanceamento com SMOTE, modelagem com XGBoost e cálculo de métricas.

In [ ]:
# Manipulação e operações numéricas
import pandas as pd
import numpy as np

# Visualização de dados
import matplotlib.pyplot as plt
import seaborn as sns

# Pré-processamento e transformação
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Pipeline compatível com reamostragem e SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# Modelo de classificação
from xgboost import XGBClassifier

# Métricas de avaliação
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    auc,
    ConfusionMatrixDisplay,
    roc_curve
)

# Configurações de exibição e gráficos
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid') if 'seaborn-v0_8-whitegrid' in plt.style.available else None
pd.set_option('display.max_columns', None)


### 5. Carregamento dos Dados
Carregamento do dataset a partir do arquivo local (`telecom_churn_synthetic.csv`).

In [ ]:
# Carregamento do arquivo local
caminho_csv = 'telecom_churn_synthetic.csv'
df = pd.read_csv(caminho_csv)
print(f"Dataset carregado com sucesso a partir de '{caminho_csv}'.")

# Visualização das primeiras linhas
df.head()

### 6. Análise Inicial e Validação dos Dados
Inspeção das dimensões, tipos de dados das colunas, contagem de valores ausentes e verificação do desbalanceamento da variável-alvo `churn`.

In [ ]:
# Dimensões da base de dados
print(f"Dimensões do dataset: {df.shape[0]} linhas e {df.shape[1]} colunas.\n")

# Informações sobre tipos e preenchimento
df.info()

In [ ]:
# Verificação de valores ausentes por coluna
missing_values = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Valores Ausentes': missing_values, 'Percentual (%)': missing_pct})
missing_df[missing_df['Valores Ausentes'] > 0]

In [ ]:
# Distribuição da variável-alvo (churn)
churn_counts = df['churn'].value_counts()
churn_pct = df['churn'].value_counts(normalize=True) * 100

print("Distribuição absoluta da classe churn:")
print(churn_counts)
print("\nDistribuição percentual da classe churn:")
print(churn_pct.round(2))

# Visualização gráfica do desbalanceamento
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x='churn', palette=['#1f77b4', '#d62728'])
plt.title('Distribuição da Variável-Alvo (0: Retido, 1: Churn)', fontsize=12)
plt.xlabel('Churn (Cancelamento)')
plt.ylabel('Quantidade de Clientes')
plt.xticks([0, 1], ['0 (Não Churn - 85.95%)', '1 (Churn - 14.05%)'])
plt.tight_layout()
plt.show()

In [ ]:
# Resumo estatístico das variáveis numéricas
df.describe()

### 7. Limpeza Inicial e Separação de Variáveis
Conversão da variável `total_charges` para formato numérico (com imputação de ausências pela mediana), separação das variáveis preditoras ($X$) e da variável-alvo ($y$), e remoção do identificador de cliente `customer_id`.

In [ ]:
# Tratamento e conversão de total_charges
df['total_charges'] = pd.to_numeric(df['total_charges'], errors='coerce')
df['total_charges'] = df['total_charges'].fillna(df['total_charges'].median())

# Separação das features (X) e do alvo (y), removendo o identificador
X = df.drop(['customer_id', 'churn'], axis=1)
y = df['churn']

print(f"Dimensões da matriz de atributos X: {X.shape}")
print(f"Dimensões do vetor alvo y: {y.shape}")

### 8. Identificação de Variáveis Numéricas e Categóricas
Mapeamento automático das colunas de acordo com o tipo de dado para direcionamento aos transformadores específicos.

In [ ]:
# Identificação das colunas categóricas e numéricas
cat_cols = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
num_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()

print(f"Variáveis Numéricas ({len(num_cols)}):\n{num_cols}\n")
print(f"Variáveis Categóricas ({len(cat_cols)}):\n{cat_cols}")

### 9. Construção do Pipeline de Pré-processamento (`ColumnTransformer`)
Definição das etapas de transformação:
- **Variáveis Numéricas:** Imputação de valores ausentes pela mediana e padronização com `StandardScaler`.
- **Variáveis Categóricas:** Imputação pelo valor mais frequente e codificação com `OneHotEncoder(handle_unknown='ignore', drop='first')`.

In [ ]:
# Pipeline de transformação para variáveis numéricas
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline de transformação para variáveis categóricas
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

# Estruturação do ColumnTransformer integrando ambos os pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

preprocessor

### 10. Divisão Manual dos Dados em Treino e Teste com Estratificação (70% / 30%)
Implementação manual da divisão estratificada sem funções prontas de partição. O algoritmo separa os índices de cada classe (`churn=0` e `churn=1`), amostra 70% de cada grupo para o conjunto de treinamento e os 30% restantes para o conjunto de teste com semente fixa (`random_state=42`), garantindo a preservação exata da proporção da classe minoritária em ambos os subconjuntos.

In [ ]:
# Proporção de divisão (70% Treino / 30% Teste) e semente de reprodutibilidade
train_ratio = 0.70
seed = 42

# Obtenção dos índices de cada classe individualmente
idx_class_0 = y[y == 0].sample(frac=1.0, random_state=seed).index
idx_class_1 = y[y == 1].sample(frac=1.0, random_state=seed).index

# Cálculo do ponto de corte proporcional (70%) para cada classe
cut_0 = int(len(idx_class_0) * train_ratio)
cut_1 = int(len(idx_class_1) * train_ratio)

# Separação dos índices para treino e teste de forma estratificada
train_idx_0, test_idx_0 = idx_class_0[:cut_0], idx_class_0[cut_0:]
train_idx_1, test_idx_1 = idx_class_1[:cut_1], idx_class_1[cut_1:]

# União dos índices de ambas as classes
train_indices = train_idx_0.union(train_idx_1)
test_indices = test_idx_0.union(test_idx_1)

# Embaralhamento dos índices combinados para evitar ordenação por classe
np.random.seed(seed)
train_indices = np.random.permutation(train_indices)
test_indices = np.random.permutation(test_indices)

# Criação dos conjuntos finais X_train, X_test, y_train e y_test
X_train, y_train = X.loc[train_indices], y.loc[train_indices]
X_test, y_test = X.loc[test_indices], y.loc[test_indices]

# Validação da estratificação manual
print(f"Total de amostras no Treino (70%): {X_train.shape[0]}")
print(f"Total de amostras no Teste  (30%): {X_test.shape[0]}\n")
print("Distribuição percentual da variável-alvo no TREINO:")
print(y_train.value_counts(normalize=True).round(4) * 100)
print("\nDistribuição percentual da variável-alvo no TESTE:")
print(y_test.value_counts(normalize=True).round(4) * 100)

### 11. Construção do `ImbPipeline` e Treinamento do Modelo XGBoost
Montagem do pipeline completo encadeando:
1. **Pré-processador:** Imputação, Padronização e One-Hot Encoding via `ColumnTransformer`.
2. **SMOTE (`random_state=42`):** Geração sintética de amostras da classe minoritária (churn) aplicada estritamente no conjunto de treino.
3. **Classificador XGBoost:** Configurado com hiperparâmetros documentados (`n_estimators=200`, `learning_rate=0.05`, `max_depth=6`, `subsample=0.8`, `colsample_bytree=0.8`, `eval_metric='logloss'`, `random_state=42`).

In [ ]:
# Definição do classificador XGBoost com parâmetros documentados
xgb_clf = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

# Estruturação do pipeline com imblearn para reamostragem segura no treino
model_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', xgb_clf)
])

# Treinamento do pipeline completo nos dados de treino
print("Iniciando o treinamento do modelo...")
model_pipeline.fit(X_train, y_train)
print("Treinamento concluído com sucesso!")

### 12. Avaliação do Modelo
Geração de previsões de classe e probabilidades no conjunto de teste, com cálculo das métricas de desempenho voltadas para cenários desbalanceados: Acurácia, ROC-AUC, PR-AUC, Relatório de Classificação (`classification_report`) e Matriz de Confusão.

In [ ]:
# Geração de predições no conjunto de teste
y_pred = model_pipeline.predict(X_test)
y_proba = model_pipeline.predict_proba(X_test)[:, 1]

# Cálculo das métricas globais e de ranking
acc = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)
precisions, recalls, _ = precision_recall_curve(y_test, y_proba)
pr_auc = auc(recalls, precisions)

print("=" * 50)
print("       MÉTRICAS DE DESEMPENHO DO MODELO")
print("=" * 50)
print(f"Acurácia:  {acc:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"PR-AUC:    {pr_auc:.4f}")
print("=" * 50)
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred, target_names=['Retido (0)', 'Churn (1)']))

In [ ]:
# Matriz de Confusão
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Retido (0)', 'Churn (1)'])
disp.plot(cmap='Blues', ax=ax, values_format='d')
plt.title('Matriz de Confusão - XGBoost com SMOTE', fontsize=12)
plt.grid(False)
plt.tight_layout()
plt.show()

print("\nDetalhamento dos Resultados da Matriz de Confusão:")
print(f"- Verdadeiros Negativos (Clientes mantidos corretamente identificados): {cm[0, 0]}")
print(f"- Falsos Positivos (Clientes mantidos classificados como churn): {cm[0, 1]}")
print(f"- Falsos Negativos (Clientes que cancelaram mas não foram detectados): {cm[1, 0]}")
print(f"- Verdadeiros Positivos (Clientes que cancelaram corretamente detectados): {cm[1, 1]}")

In [ ]:
# Curvas ROC e Precision-Recall
fpr, tpr, _ = roc_curve(y_test, y_proba)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Curva ROC
axes[0].plot(fpr, tpr, color='#1f77b4', lw=2, label=f'XGBoost (AUC = {roc_auc:.4f})')
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1, label='Aleatório (AUC = 0.5000)')
axes[0].set_title('Curva ROC (Receiver Operating Characteristic)', fontsize=12)
axes[0].set_xlabel('Taxa de Falsos Positivos (1 - Especificidade)')
axes[0].set_ylabel('Taxa de Verdadeiros Positivos (Recall / Sensibilidade)')
axes[0].legend(loc='lower right')

# Curva Precision-Recall
baseline_pr = y_test.mean()
axes[1].plot(recalls, precisions, color='#d62728', lw=2, label=f'XGBoost (PR-AUC = {pr_auc:.4f})')
axes[1].axhline(y=baseline_pr, color='gray', linestyle='--', lw=1, label=f'Linha de Base ({baseline_pr:.4f})')
axes[1].set_title('Curva Precision-Recall', fontsize=12)
axes[1].set_xlabel('Recall (Revocação)')
axes[1].set_ylabel('Precision (Precisão)')
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

### 13. Análise de Importância das Variáveis (*Feature Importance*)
Inspeção dos atributos mais relevantes utilizados pelo modelo XGBoost para a tomada de decisão.

In [ ]:
# Obtenção dos nomes das features após o pré-processamento (One-Hot Encoding)
ohe_feature_names = model_pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(cat_cols)
all_feature_names = num_cols + list(ohe_feature_names)

# Extração das importâncias do XGBoost
importances = model_pipeline.named_steps['classifier'].feature_importances_
feat_imp_df = pd.DataFrame({'Atributo': all_feature_names, 'Importancia': importances})
feat_imp_df = feat_imp_df.sort_values(by='Importancia', ascending=False).head(15)

# Visualização gráfica
plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp_df, x='Importancia', y='Atributo', palette='viridis')
plt.title('Top 15 Variáveis Mais Importantes - XGBoost', fontsize=12)
plt.xlabel('Importância Relativa')
plt.ylabel('Atributo')
plt.tight_layout()
plt.show()

### 14. Interpretação Técnica e Análise Crítica dos Resultados

#### 14.1 Diagnóstico de Desempenho
- **Acurácia aparentemente elevada (~85,9%):** A acurácia é uma métrica ilusória em bases desbalanceadas. Como ~85,95% dos clientes pertencem à classe 0 (não churn), um modelo ingênuo que classificasse todos os clientes como "não churn" alcançaria acurácia semelhante sem identificar nenhum cliente em risco.
- **Recall Crítico da Classe Churn (~6,5% a 7,0%):** O modelo identificou apenas uma pequena fração dos clientes que realmente cancelaram no conjunto de teste. Isso evidencia que a grande maioria dos clientes propensos ao churn não foi detectada pelo limiar padrão (0.50).
- **ROC-AUC e PR-AUC:** O ROC-AUC indica capacidade moderada de ordenação probabilística, mas o PR-AUC reflete a real dificuldade de precisão sobre a classe minoritária.

#### 14.2 Hipótese de Teto Informacional
Conforme documentado nos experimentos anteriores (Regressão Logística, Gradient Boosting, LightGBM, SMOTE e ajuste de pesos), os modelos atingem desempenho próximo (ROC-AUC entre 0,71 e 0,75; PR-AUC entre 0,28 e 0,34).
Esse comportamento aponta para um **teto informacional da base de dados**, sugerindo que as variáveis preditoras atuais possuem correlação limitada com os determinantes reais de cancelamento.

### 15. Insights de Negócio e Estratégias Práticas de Retenção

A análise exploratória e os padrões extraídos da modelagem revelam diretrizes práticas para a tomada de decisão e redução da perda de receita:

#### 15.1 Calibração de Decisão pelo Custo de Negócio (*Threshold Tuning*)
- **O Problema do Limiar Padrão (0.50):** Ao exigir 50% de probabilidade para agir, a empresa deixa escapar ~93% dos clientes que cancelam (falsos negativos).
- **Assimetria de Custos (CAC vs Retenção):** Conquistar um novo cliente (CAC) custa de 5 a 7 vezes mais do que retê-lo com uma ação comercial de baixo custo (ex: cupom, suporte técnico gratuito, upgrade temporário de velocidade).
- **Ação Recomendada:** Reduzir o limiar de decisão operacional para a faixa entre **0.20 e 0.30**. Isso aumenta a captura de cancelamentos reais (*Recall* de ~50-70%) a um custo aceitável de falsos positivos.

#### 15.2 Matriz de Ações Preventivas por Fator de Risco
1. **Risco por Experiência e Qualidade do Serviço (`complaints_last_3m` e `outages_last_3m`):**
   - *Diagnóstico:* Clientes com reclamações recentes ou histórico de quedas de rede têm risco exponencial de evasão.
   - *Ação:* Criação de uma **esteira de intervenção rápida (SAC Proativo)**: clientes com mais de 1 queda ou chamado aberto no mês são direcionados para atendimento VIP e recebem compensação/desconto na fatura antes de solicitarem o cancelamento.
2. **Risco Contratual e Tempo de Casa (`contract` e `tenure_months`):**
   - *Diagnóstico:* Clientes em contratos mensais (*Month-to-Month*) e nos primeiros 6 a 12 meses apresentam a maior taxa de evasão.
   - *Ação:* Campanhas de incentivo para migração para **contratos anuais ou bienais** com descontos progressivos e inclusão de benefícios agregados (ex: *streaming* ou antivírus/backup inclusos).
3. **Serviços de Suporte como Âncora de Retenção (`tech_support` e `online_security`):**
   - *Diagnóstico:* Clientes que não contratam serviços de proteção e suporte desistem com maior facilidade perante instabilidades técnicas.
   - *Ação:* Oferecer pacotes combinados (*bundles*) onde suporte técnico e proteção de dispositivos venham bonificados nos planos de internet de alta velocidade.
4. **Risco Financeiro e Forma de Pagamento (`late_payments_last_6m` e `payment_method`):**
   - *Diagnóstico:* Clientes com atrasos de fatura e métodos de pagamento manuais (boleto/cheque) têm maior atrito financeiro.
   - *Ação:* Oferecer bonificação mensal ou desconto fixo na fatura para clientes que migrarem para **débito automático** ou cartão de crédito em fatura digital (*paperless billing*).

#### 15.3 Segmentação Operacional em Decis de Risco (Tiering)
O score probabilístico gerado pelo modelo (`y_proba`) deve ser usado para dividir a base de clientes em 3 níveis operacionais:
- **Tier 1 - Alto Risco / Alto LTV (Top 10% do score):** Ação humana de alto contato (gerente de conta / equipe de retenção dedicada com ofertas personalizadas).
- **Tier 2 - Médio Risco (Top 10% a 30% do score):** Ações automatizadas via CRM (e-mails com benefícios, melhoria de velocidade sem custo adicional, pesquisa de satisfação NPS).
- **Tier 3 - Baixo Risco (Demais 70%):** Comunicação padrão institucional e ações de fidelização contínua.

### 16. Limitações e Próximos Passos

#### Limitações Identificadas:
1. **Ausência de Dados Comportamentais Dinâmicos:** Variáveis estáticas ou agregadas de curto prazo (últimos 3 a 6 meses) limitam a captura de tendências e deterioração súbita no uso do serviço.
2. **Trade-off Precision/Recall Estático:** O uso do limiar padrão de 50% penalizou severamente o recall da classe de maior interesse de negócio.

#### Próximos Passos Recomendados:
1. **Otimização do Limiar de Decisão (*Threshold Tuning*):** Avaliar curvas de custo-benefício financeiro (custo de reter falsos positivos vs. custo da perda por falsos negativos) para fixar um limiar ideal (ex: entre 0.20 e 0.35).
2. **Enriquecimento da Base de Dados:** Adicionar métricas de engajamento do cliente, histórico de chamados no SAC em tempo real, NPS recente e padrão detalhado de tráfego de dados.
3. **Testes de Engenharia de Features Avançada:** Criação de variáveis de razão (ex: proporção do valor mensal sobre o total acumulado) e análise detalhada com SHAP (*SHapley Additive exPlanations*).
4. **Exportação do Pipeline:** Persistência do objeto `model_pipeline` para reprodutibilidade e esteira de inferência.

### 17. Conclusão e Síntese Final
O pipeline de Machine Learning construído seguiu rigorosamente as melhores práticas metodológicas:
- Limpeza e tratamento adequado de nulos sem vazamento de dados.
- Padronização e codificação categórica acopladas via `ColumnTransformer`.
- Divisão estratificada manual (70% treino / 30% teste).
- Reamostragem com SMOTE restrita exclusivamente ao conjunto de treino através do `ImbPipeline`.
- Avaliação e cálculo de métricas específicas para classes raras.

O experimento demonstra consistência técnica e rigor acadêmico, fornecendo uma base sólida para evolução do projeto por meio de engenharia de atributos e calibração de decisões estratégicas de negócio.